# Feature selector comparison

Which selector should the age model keep?

`|corr|` top-150 stays unless another set beats it by at least 0.01 outer-fold XGBoost R² and by more than the fold standard error. This notebook reads the saved run. It does not retrain.


In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from experiments.blocks import report

payload = json.loads((ROOT / "experiments/001/results/k150_matrix.json").read_text())
verdict = payload["verdict"]
rows = []
for name, row in verdict["table"].items():
    rows.append({
        "set": name,
        "p": int(pd.Series(row["n_features"]).median()),
        "ridge": row["ridge"]["r2_mean"],
        "xgb": row["xgb"]["r2_mean"],
        "svr": row["svr"]["r2_mean"],
        "xgb_rmse": row["xgb"]["rmse_mean"],
    })
scores = pd.DataFrame(rows).sort_values("xgb", ascending=False)
detail = verdict["h1a_detail"]
print(
    f"keep {verdict['keep']}    "
    f"{detail['selector']} vs all = {detail['mean']:+.3f} (SE {detail['se']:.3f})    "
    f"H1a {verdict['h1a']}    H1b {verdict['h1b']}"
)
report(scores, "Outer-fold mean R²", bar="xgb")


keep corr    corr vs all = +0.043 (SE 0.009)    H1a supported    H1b not_supported


set,p,ridge,xgb,svr,xgb_rmse,
corr,150,0.289,0.543,0.424,6.549,
elastic_net,150,0.262,0.505,0.372,6.804,
mi,150,0.269,0.503,0.401,6.829,
xgb_perm,150,0.269,0.500,0.378,6.852,
all,825,-2.621,0.499,0.246,6.850,
monte_carlo,150,0.192,0.425,0.251,7.344,


## Folds

Same five splits for every set.


In [2]:
fold_rows = []
for name in scores["set"]:
    rec = {"set": name}
    for i, value in enumerate(verdict["table"][name]["xgb"]["r2_folds"], start=1):
        rec[f"f{i}"] = value
    fold_rows.append(rec)
report(pd.DataFrame(fold_rows), "XGBoost R² on each fold")


set,f1,f2,f3,f4,f5
corr,0.556,0.526,0.494,0.597,0.540
elastic_net,0.537,0.509,0.398,0.575,0.509
mi,0.515,0.481,0.446,0.561,0.509
xgb_perm,0.522,0.502,0.434,0.541,0.500
all,0.534,0.481,0.419,0.556,0.506
monte_carlo,0.405,0.373,0.421,0.441,0.486


## Do the selectors pick the same columns?

Stability is the overlap of a selector with itself across folds. Overlap is against `|corr|` inside a fold. 1 means the same 150 columns.


In [3]:
rows = []
for name in scores["set"]:
    if name == "all":
        continue
    rows.append({
        "set": name,
        "stability": verdict["stability_jaccard"][name],
        "overlap_with_corr": 1.0 if name == "corr" else verdict["jaccard_vs_corr"][name],
    })
report(pd.DataFrame(rows), "Column agreement", bar="stability")


set,stability,overlap_with_corr,
corr,0.931,1.000,
elastic_net,0.565,0.159,
mi,0.446,0.443,
xgb_perm,0.199,0.252,
monte_carlo,0.108,0.115,
